# 01 — Preprocessing

**Primary:** Tuan Wei  
**Support:** Tianyi Qin

**RQ:** Among Melbourne entire homes and apartments with valid nightly prices, do location and amenities improve high-price prediction beyond property size alone, and which attributes are most useful?

This notebook uses the **original Melbourne Detailed Listings dataset downloaded directly from Inside Airbnb**. It does not use the cleaned/modified Assignment 1 dataset.

## 1. Imports and official data source

In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

REPO_ROOT = Path("..").resolve() if Path("../src").exists() else Path(".").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data_source import ensure_melbourne_listings, MELBOURNE_LISTINGS_GZ_URL
from src.a1_reuse import (
    validate_a2_source,
    validate_melbourne_source,
    add_clean_price,
    add_amenity_count,
    add_bathrooms_numeric,
    parse_amenities,
)

DATA_PATH = ensure_melbourne_listings(REPO_ROOT / "data" / "listings.csv")
print("Official source:", MELBOURNE_LISTINGS_GZ_URL)
print("Using:", DATA_PATH.resolve())

## 2. Load and validate the raw original dataset

The two guards below reject the known 29-column A1 teaching dataset and a clearly non-Melbourne file.

In [ ]:
df_raw = pd.read_csv(DATA_PATH, low_memory=False)
validate_a2_source(df_raw)
validate_melbourne_source(df_raw)

print("Raw shape:", df_raw.shape)
print("Median coordinates:",
      round(pd.to_numeric(df_raw["latitude"], errors="coerce").median(), 5),
      round(pd.to_numeric(df_raw["longitude"], errors="coerce").median(), 5))
display(df_raw.head())

## 3. Initial audit

In [ ]:
audit = pd.DataFrame({
    "dtype": df_raw.dtypes.astype(str),
    "missing_n": df_raw.isna().sum(),
    "missing_pct": (df_raw.isna().mean() * 100).round(2),
    "n_unique": df_raw.nunique(dropna=True),
}).sort_values(["missing_pct", "missing_n"], ascending=False)

display(audit.head(50))

## 4. Define the RQ cohort

The analysis population is restricted to:
- `room_type == "Entire home/apt"`;
- a valid, finite, positive nightly price.

Price parsing reuses the group's A1 corrected parsing logic, but it is rerun on the original A2 data.

In [ ]:
n_raw = len(df_raw)

df = df_raw.loc[df_raw["room_type"].eq("Entire home/apt")].copy()
n_entire = len(df)

df = add_clean_price(df, source="price", target="price_clean")
valid_price = (
    df["price_clean"].notna()
    & np.isfinite(df["price_clean"])
    & df["price_clean"].gt(0)
)
df = df.loc[valid_price].copy()
n_eligible = len(df)

cohort_summary = pd.DataFrame({
    "stage": [
        "Raw original listings",
        "Entire home/apt",
        "Entire home/apt + valid positive nightly price",
    ],
    "rows": [n_raw, n_entire, n_eligible],
})
cohort_summary["retained_pct_of_raw"] = (
    cohort_summary["rows"] / n_raw * 100
).round(2)

display(cohort_summary)
display(df["price_clean"].describe(percentiles=[.25,.5,.75,.9,.95,.99]))

## 5. Six preprocessing candidates

The assignment requires at least six candidates and three selected tasks. The final three are chosen to map directly to the RQ's three constructs: **property size, location, and amenities**.

Selected:
1. Reuse A1 numeric bathrooms derived from `bathrooms_text`.
2. Engineer distance from Melbourne CBD from latitude/longitude.
3. Engineer amenity count and a small set of amenity indicators.

Not selected as separate preprocessing tasks:
4. Special missing-value treatment for bedrooms/beds.
5. Property-type consolidation.
6. Neighbourhood encoding/consolidation.

The table below generates the dataset-specific evidence used to justify these choices.

## 6. Selected task 1 — A1 numeric bathrooms

In [ ]:
# Reuse the A1 pipeline logic as required by the A2 specification.
df = add_bathrooms_numeric(
    df,
    source="bathrooms_text",
    target="bathrooms",
)

bathroom_summary = {
    "rows": len(df),
    "source_text_categories": int(df["bathrooms_text"].nunique(dropna=True)),
    "source_missing_n": int(df["bathrooms_text"].isna().sum()),
    "numeric_nonmissing_n": int(df["bathrooms"].notna().sum()),
    "numeric_missing_n": int(df["bathrooms"].isna().sum()),
    "numeric_usable_pct": round(float(df["bathrooms"].notna().mean() * 100), 2),
}
bathroom_summary

## 7. Selected task 2 — distance from Melbourne CBD

Distance is calculated with the Haversine formula using approximately `(-37.8136, 144.9631)` as the CBD reference point.

In [ ]:
def haversine_km(lat, lon, ref_lat=-37.8136, ref_lon=144.9631):
    lat1 = np.radians(pd.to_numeric(lat, errors="coerce"))
    lon1 = np.radians(pd.to_numeric(lon, errors="coerce"))
    lat2 = np.radians(ref_lat)
    lon2 = np.radians(ref_lon)

    dlat = lat1 - lat2
    dlon = lon1 - lon2
    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )
    return 6371.0088 * 2 * np.arcsin(np.sqrt(a))

df["distance_cbd_km"] = haversine_km(df["latitude"], df["longitude"])
display(df["distance_cbd_km"].describe(percentiles=[.25,.5,.75,.9,.95]))

## 8. Selected task 3 — amenities engineering

A1 showed why naive comma splitting is unsafe for this field. The A1 parser is reused on the original A2 data. The task creates:
- `amenity_count`;
- six interpretable amenity indicators for the full prediction model.

In [ ]:
df = add_amenity_count(df, source="amenities", target="amenity_count")
df["amenities_list"] = df["amenities"].apply(parse_amenities)

AMENITY_KEYWORDS = {
    "has_pool": ("pool",),
    "has_free_parking": ("free parking",),
    "has_air_conditioning": ("air conditioning", "window ac", "central air"),
    "has_kitchen": ("kitchen",),
    "has_washer": ("washer",),
    "has_dryer": ("dryer",),
}

def amenity_indicator(items, keywords):
    text = " | ".join(str(x).lower() for x in items)
    return int(any(keyword in text for keyword in keywords))

for feature, keywords in AMENITY_KEYWORDS.items():
    df[feature] = df["amenities_list"].apply(
        lambda items, kw=keywords: amenity_indicator(items, kw)
    )

amenity_indicator_summary = pd.DataFrame({
    "feature": list(AMENITY_KEYWORDS),
    "n_yes": [int(df[c].sum()) for c in AMENITY_KEYWORDS],
    "pct_yes": [round(float(df[c].mean() * 100), 2) for c in AMENITY_KEYWORDS],
})
display(amenity_indicator_summary)
display(df["amenity_count"].describe(percentiles=[.25,.5,.75,.9,.95]))

## 9. Dataset-specific candidate decision table

The rejected candidates are still handled where necessary:
- bedrooms/beds missingness is handled inside model pipelines with training-fold median imputation;
- property type is not added because it is outside the RQ's size/location/amenity comparison;
- neighbourhood encoding is not used because distance provides a compact location feature without 30 dummy variables.

In [ ]:
property_counts = df["property_type"].value_counts(dropna=False)
neighbour_counts = df["neighbourhood_cleansed"].value_counts(dropna=False)

bedrooms_missing_n = int(df["bedrooms"].isna().sum())
beds_missing_n = int(df["beds"].isna().sum())

candidate_decisions = pd.DataFrame([
    {
        "candidate": "Numeric bathrooms from bathrooms_text (A1 pipeline)",
        "selected": True,
        "dataset_evidence": (
            f'{bathroom_summary["numeric_nonmissing_n"]:,}/{len(df):,} rows '
            f'({bathroom_summary["numeric_usable_pct"]:.2f}%) receive a numeric value '
            f'from {bathroom_summary["source_text_categories"]} text categories.'
        ),
    },
    {
        "candidate": "Distance from CBD from latitude/longitude",
        "selected": True,
        "dataset_evidence": (
            f'{int(df["distance_cbd_km"].notna().sum()):,}/{len(df):,} rows receive distance; '
            f'median={df["distance_cbd_km"].median():.2f} km.'
        ),
    },
    {
        "candidate": "Amenity count + amenity indicators",
        "selected": True,
        "dataset_evidence": (
            f'Raw amenities has {df["amenities"].nunique(dropna=True):,} unique strings; '
            f'amenity_count has {df["amenity_count"].nunique(dropna=True)} unique values '
            f'across {len(df):,} rows.'
        ),
    },
    {
        "candidate": "Special bedroom/bed missing-value preprocessing",
        "selected": False,
        "dataset_evidence": (
            f'Bedrooms missing: {bedrooms_missing_n:,} '
            f'({bedrooms_missing_n/len(df)*100:.2f}%); beds missing: {beds_missing_n:,} '
            f'({beds_missing_n/len(df)*100:.2f}%). Median imputation is kept inside model CV.'
        ),
    },
    {
        "candidate": "Property-type consolidation",
        "selected": False,
        "dataset_evidence": (
            f'{property_counts.size} property types; '
            f'{int((property_counts < 10).sum())} have fewer than 10 eligible listings. '
            'Excluded to keep the RQ comparison focused on size, location and amenities.'
        ),
    },
    {
        "candidate": "Neighbourhood encoding/consolidation",
        "selected": False,
        "dataset_evidence": (
            f'{neighbour_counts.size} neighbourhood categories with '
            f'{int(df["neighbourhood_cleansed"].isna().sum())} missing rows. '
            'Distance-to-CBD is used as the primary location representation.'
        ),
    },
])

display(candidate_decisions)

## 10. Measurable impact of the three selected preprocessing tasks

In [ ]:
preprocessing_impact = pd.DataFrame([
    {
        "task": "A1 numeric bathrooms",
        "before": f'{df["bathrooms_text"].nunique(dropna=True)} text categories',
        "after": f'{df["bathrooms"].notna().sum():,} numeric values',
        "rows_affected": int(df["bathrooms"].notna().sum()),
    },
    {
        "task": "Distance from CBD",
        "before": "latitude + longitude as two coordinates",
        "after": (
            f'one distance feature; median={df["distance_cbd_km"].median():.2f} km, '
            f'max={df["distance_cbd_km"].max():.2f} km'
        ),
        "rows_affected": int(df["distance_cbd_km"].notna().sum()),
    },
    {
        "task": "Amenities engineering",
        "before": f'{df["amenities"].nunique(dropna=True):,} unique raw amenity strings',
        "after": (
            f'{df["amenity_count"].nunique(dropna=True)} amenity-count values + '
            f'{len(AMENITY_KEYWORDS)} binary amenity indicators'
        ),
        "rows_affected": int(len(df)),
    },
])

display(preprocessing_impact)

## 11. Shared target and split definition

The group contract defines **high price** as price above the **training sample's 75th percentile**.

To also satisfy the rubric's stratified-split requirement without using test prices to set the threshold, the code iterates:
1. stratify using the current threshold;
2. compute Q75 from the resulting training sample only;
3. repeat until the threshold is unchanged.

At convergence, the split is stratified by the same labels produced by the final training-sample threshold.

In [ ]:
def make_training_q75_stratified_split(
    frame,
    price_col="price_clean",
    test_size=0.20,
    random_state=42,
    max_iter=50,
    atol=1e-10,
):
    indices = np.arange(len(frame))
    threshold = float(frame[price_col].quantile(0.75))
    history = []

    for iteration in range(1, max_iter + 1):
        labels = (frame[price_col].to_numpy() > threshold).astype(int)

        train_idx, test_idx = train_test_split(
            indices,
            test_size=test_size,
            random_state=random_state,
            stratify=labels,
        )

        new_threshold = float(
            frame.iloc[train_idx][price_col].quantile(0.75)
        )
        history.append({
            "iteration": iteration,
            "threshold_used_for_stratification": threshold,
            "training_q75": new_threshold,
        })

        if np.isclose(new_threshold, threshold, rtol=0, atol=atol):
            final_labels = (
                frame[price_col].to_numpy() > new_threshold
            ).astype(int)
            return train_idx, test_idx, new_threshold, final_labels, pd.DataFrame(history)

        threshold = new_threshold

    raise RuntimeError(
        "Training-Q75 / stratified-split iteration did not converge. "
        "Review the split design with the tutor."
    )

train_idx, test_idx, price_threshold, target, split_history = (
    make_training_q75_stratified_split(
        df,
        random_state=RANDOM_STATE,
    )
)

df["high_price"] = target
df["split"] = "test"
df.iloc[train_idx, df.columns.get_loc("split")] = "train"

display(split_history)

split_summary = (
    df.groupby(["split", "high_price"])
      .size()
      .rename("n")
      .reset_index()
)
split_summary["pct_within_split"] = (
    split_summary["n"]
    / split_summary.groupby("split")["n"].transform("sum")
    * 100
).round(2)

print("Final training-sample Q75 threshold:", price_threshold)
display(split_summary)

## 12. Export the stable handoff dataset and preprocessing evidence

All later notebooks use this exact file so target, split, rows and feature engineering remain consistent.

In [ ]:
SIZE_FEATURES = ["accommodates", "bedrooms", "beds", "bathrooms"]
LOCATION_FEATURES = ["distance_cbd_km"]
AMENITY_FEATURES = [
    "amenity_count",
    "has_pool",
    "has_free_parking",
    "has_air_conditioning",
    "has_kitchen",
    "has_washer",
    "has_dryer",
]

export_cols = [
    "id",
    "price_clean",
    "high_price",
    "split",
    "property_type",
    "neighbourhood_cleansed",
    "latitude",
    "longitude",
] + SIZE_FEATURES + LOCATION_FEATURES + AMENITY_FEATURES

processed = df[export_cols].copy()

DATA_OUT = REPO_ROOT / "data" / "processed_listings.csv"
TABLE_OUT = REPO_ROOT / "output" / "tables"
FIG_OUT = REPO_ROOT / "output" / "figures"
TABLE_OUT.mkdir(parents=True, exist_ok=True)
FIG_OUT.mkdir(parents=True, exist_ok=True)

processed.to_csv(DATA_OUT, index=False)
candidate_decisions.to_csv(TABLE_OUT / "preprocessing_candidates.csv", index=False)
preprocessing_impact.to_csv(TABLE_OUT / "preprocessing_impact.csv", index=False)
split_summary.to_csv(TABLE_OUT / "split_summary.csv", index=False)
amenity_indicator_summary.to_csv(TABLE_OUT / "amenity_indicator_prevalence.csv", index=False)

metadata = {
    "source_url": MELBOURNE_LISTINGS_GZ_URL,
    "raw_rows": int(n_raw),
    "entire_home_rows": int(n_entire),
    "eligible_rows": int(n_eligible),
    "training_price_q75": float(price_threshold),
    "train_rows": int((processed["split"] == "train").sum()),
    "test_rows": int((processed["split"] == "test").sum()),
    "size_features": SIZE_FEATURES,
    "location_features": LOCATION_FEATURES,
    "amenity_features": AMENITY_FEATURES,
}
(TABLE_OUT / "preprocessing_metadata.json").write_text(
    json.dumps(metadata, indent=2),
    encoding="utf-8",
)

print("Saved:", DATA_OUT)
print("Processed shape:", processed.shape)
display(processed.head())

## 13. Simple data visualisations for later report selection

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.hist(df["price_clean"].clip(upper=df["price_clean"].quantile(0.99)), bins=40)
ax.axvline(price_threshold, linestyle="--", linewidth=1.5)
ax.set_xlabel("Nightly price (AUD; clipped at 99th percentile for display)")
ax.set_ylabel("Listings")
ax.set_title("Eligible Melbourne entire-home nightly prices")
fig.tight_layout()
fig.savefig(FIG_OUT / "eligible_price_distribution.png", dpi=200)
plt.close(fig)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.hist(df["distance_cbd_km"], bins=40)
ax.set_xlabel("Distance from Melbourne CBD (km)")
ax.set_ylabel("Listings")
ax.set_title("Distance-to-CBD distribution")
fig.tight_layout()
fig.savefig(FIG_OUT / "distance_cbd_distribution.png", dpi=200)
plt.close(fig)

print("Saved preprocessing figures to:", FIG_OUT)